# FinDrama LOB Pretraining

Works on **H100, L4, A100, T4** - any CUDA GPU. Detects architecture at build time and compiles mamba-ssm kernels accordingly.

In [ ]:
# Edit these before running.
REPO_URL = "https://github.com/Ruuudy1/FinDrama.git"
BRANCH = "improvements-research-review"
PROJECT_DIR = "/content/Drama"

# Reuse compiled CUDA wheels and pull the dataset bundle from Hugging Face Hub.
# Add HF_TOKEN to Colab Secrets (key icon, left sidebar) before running.
HF_REPO = "ruuudy/FinDrama"
HF_REVISION = None
FORCE_REBUILD_WHEELS = False

# Use True for a dependency and data smoke test before the full run.
SMOKE_TEST = False
HOURS_TRAIN = 6
HOURS_VAL = 1

# RUN_PROBES True runs the three collapse-hypothesis probes (H1, H2, H3) at
# PROBE_STEPS each. Set to False to run a single full pretrain at MAX_STEPS.
RUN_PROBES = True
PROBE_STEPS = 20 if SMOKE_TEST else 1000
MAX_STEPS = 20 if SMOKE_TEST else 20000

In [ ]:
import os
from pathlib import Path
import subprocess

# Resolve the HF token from Colab Secrets first, then env var.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN missing. Add it to Colab Secrets or export it before running.")

subprocess.check_call(["pip", "install", "-q", "huggingface_hub"])
from huggingface_hub import snapshot_download

DATA_DOWNLOAD_DIR = Path("/content/findrama_data")
DATA_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Pulling dataset bundle from HuggingFace Hub.")
snapshot_download(
    repo_id=HF_REPO,
    repo_type="dataset",
    allow_patterns=["data/train.tar.zip", "data/validation.tar.zip"],
    revision=HF_REVISION,
    token=HF_TOKEN,
    local_dir=str(DATA_DOWNLOAD_DIR),
)
TRAIN_ZIP = DATA_DOWNLOAD_DIR / "data" / "train.tar.zip"
VAL_ZIP = DATA_DOWNLOAD_DIR / "data" / "validation.tar.zip"
for p in (TRAIN_ZIP, VAL_ZIP):
    if not p.exists():
        raise FileNotFoundError(p)
print("Dataset zips ready:", TRAIN_ZIP.name, VAL_ZIP.name)

In [ ]:
import os, shutil, subprocess

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR])
os.chdir(PROJECT_DIR)
print("Repo ready:", os.getcwd())

In [ ]:
from pathlib import Path
import os, shlex, subprocess, sys

os.chdir(PROJECT_DIR)

def run(cmd, env=None):
    print("\n$", cmd)
    subprocess.check_call(cmd, shell=True, env=env)

def q(path):
    return shlex.quote(str(path))

cache_root = Path("/content/findrama_colab_cache")
pip_cache = cache_root / "pip"
pip_cache.mkdir(parents=True, exist_ok=True)
os.environ["PIP_CACHE_DIR"] = str(pip_cache)
print("Pip cache:", pip_cache)

run("pip install -q --upgrade pip")
run("pip install -q 'huggingface_hub[cli]'")
run("pip install -q packaging ninja setuptools==69.5.1 'numpy>=2,<3'")
run("pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124")

import torch
cuda_version = (torch.version.cuda or "cpu").replace(".", "")
gpu_name = "cpu"
arch_list = ""
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    arch_list = f"{major}.{minor}"
    gpu_name = f"sm{major}{minor}"

wheelhouse = cache_root / f"wheels-py{sys.version_info.major}{sys.version_info.minor}-torch260-cu{cuda_version}-{gpu_name}"
wheelhouse.mkdir(parents=True, exist_ok=True)
if arch_list:
    os.environ["TORCH_CUDA_ARCH_LIST"] = arch_list
print("Wheel cache:", wheelhouse)
print("TORCH_CUDA_ARCH_LIST:", os.environ.get("TORCH_CUDA_ARCH_LIST", ""))

# Pull cached wheels from HF Hub (anonymous for public repo; upload needs HF_TOKEN secret).
HF_TOKEN = None
if HF_REPO:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
    if not HF_TOKEN:
        HF_TOKEN = os.environ.get('HF_TOKEN')
    try:
        from huggingface_hub import snapshot_download
        print(f"Pulling cached wheels from HF Hub ({wheelhouse.name}) ...")
        snapshot_download(
            repo_id=HF_REPO,
            repo_type="dataset",
            allow_patterns=f"{wheelhouse.name}/*.whl",
            local_dir=str(cache_root),
            token=HF_TOKEN,
        )
        pulled = list(wheelhouse.glob("*.whl"))
        print("Wheels after pull:", [p.name for p in pulled] or ["none - will build"])
    except Exception as e:
        print(f"HF Hub pull skipped: {e}")

def latest_wheel(prefix):
    wheels = sorted(wheelhouse.glob(f"{prefix}-*.whl"), key=lambda p: p.stat().st_mtime, reverse=True)
    return wheels[0] if wheels else None

def _upload_to_hf(wheel):
    if not HF_REPO or not HF_TOKEN:
        if HF_REPO and not HF_TOKEN:
            print("HF_TOKEN not set - wheel saved locally only. Add it to Colab Secrets to persist.")
        return
    try:
        from huggingface_hub import HfApi
        HfApi().upload_file(
            path_or_fileobj=str(wheel),
            path_in_repo=f"{wheelhouse.name}/{wheel.name}",
            repo_id=HF_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        print(f"Uploaded wheel to HF Hub: {wheel.name}")
    except Exception as e:
        print(f"HF Hub upload failed (wheel saved locally only): {e}")

def build_or_install_cached(prefix, build_cmd, package_name):
    wheel = latest_wheel(prefix)
    if FORCE_REBUILD_WHEELS or wheel is None:
        if FORCE_REBUILD_WHEELS and wheel is not None:
            print("Ignoring cached wheel due to FORCE_REBUILD_WHEELS:", wheel.name)
        print(f"Building {package_name} wheel into cache. First run can be slow.")
        run(build_cmd)
        wheel = latest_wheel(prefix)
        if wheel is None:
            raise FileNotFoundError(f"No {prefix} wheel found in {wheelhouse}")
        _upload_to_hf(wheel)
    else:
        print(f"Using cached {package_name} wheel:", wheel.name)
    run(f"pip install -q --force-reinstall --no-deps {q(wheel)}")

build_or_install_cached(
    "causal_conv1d",
    f"pip wheel -q --no-deps --no-build-isolation --wheel-dir {q(wheelhouse)} 'causal-conv1d>=1.4.0'",
    "causal-conv1d",
)
build_or_install_cached(
    "mamba_ssm",
    f"MAMBA_FORCE_BUILD=TRUE pip wheel -q --no-deps --no-build-isolation --wheel-dir {q(wheelhouse)} git+https://github.com/state-spaces/mamba.git",
    "mamba-ssm",
)
run("pip install -q -r requirements.txt")
run("pip install -q tilelang")
run("pip install -q --upgrade triton")

import numpy as np
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)
print("numpy", np.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

import triton
if not hasattr(triton, 'set_allocator'):
    # Pytorch-triton lacks triton.set_allocator; this stub allows mamba_ssm to import.
    # TMA descriptor memory falls back to pytorch-triton's internal allocator.
    triton.set_allocator = lambda fn: None

import causal_conv1d_cuda
from mamba_ssm.modules.mamba3 import Mamba3
print("Mamba3 imported:", Mamba3)

try:
    from mamba_ssm.ops.tilelang.mamba3.mamba3_mimo import mamba3_mimo
except Exception as exc:
    print("Mamba3 MIMO TileLang import: unavailable", repr(exc))
else:
    print("Mamba3 MIMO TileLang import: ok", mamba3_mimo)

In [ ]:
import shutil
import tarfile
import zipfile
from pathlib import Path

project = Path(PROJECT_DIR)
data_dir = project / "data"
train_dir = data_dir / "train"
val_dir = data_dir / "validation"

def _is_metadata_path(path):
    parts = [str(x) for x in Path(path).parts]
    return any(part == "__MACOSX" or part.startswith("._") for part in parts)

def _extract_zip(zip_path, raw):
    print("Extracting", zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            if member.is_dir() or _is_metadata_path(member.filename):
                continue
            zf.extract(member, raw)

def _safe_extract_tar(archive, out):
    if _is_metadata_path(archive) or not tarfile.is_tarfile(archive):
        print("Skipping non-tar archive:", archive)
        return
    out.mkdir(exist_ok=True)
    print("Extracting", archive)
    with tarfile.open(archive) as tf:
        members = [m for m in tf.getmembers() if not _is_metadata_path(m.name)]
        try:
            tf.extractall(out, members=members, filter="data")
        except TypeError:
            tf.extractall(out, members=members)

def extract_bundle():
    if (train_dir / "polymarket.db").exists() and (val_dir / "polymarket.db").exists():
        print("Data already prepared at", data_dir)
        return
    raw = Path("/content/findrama_data_raw")
    if raw.exists():
        shutil.rmtree(raw)
    raw.mkdir(parents=True)
    for zip_path in (TRAIN_ZIP, VAL_ZIP):
        _extract_zip(zip_path, raw)
    for archive in list(raw.rglob("*.tar")) + list(raw.rglob("*.tar.gz")) + list(raw.rglob("*.tgz")):
        out = raw / archive.stem.replace(".tar", "")
        _safe_extract_tar(archive, out)
    split_dirs = [p.parent for p in raw.rglob("polymarket.db") if not _is_metadata_path(p)]
    if not split_dirs:
        raise RuntimeError("Could not find polymarket.db after extraction.")
    train_src = next((p for p in split_dirs if "train" in str(p).lower()), split_dirs[0])
    val_src = next((p for p in split_dirs if "validation" in str(p).lower() or "val" in str(p).lower()), None)
    if val_src is None:
        raise RuntimeError("Could not identify validation split.")
    data_dir.mkdir(exist_ok=True)
    shutil.rmtree(train_dir, ignore_errors=True)
    shutil.rmtree(val_dir, ignore_errors=True)
    shutil.copytree(train_src, train_dir)
    shutil.copytree(val_src, val_dir)
    print("Prepared train:", train_dir)
    print("Prepared validation:", val_dir)

extract_bundle()

In [ ]:
import subprocess, os, sys

os.chdir(PROJECT_DIR)

# Each probe flips one hardcoded constant from default to test a collapse hypothesis.
# H1 raises the representation KL weight to match DreamerV3 0.5/0.5 symmetry.
# H2 zeros the free-bits floor to free the dynamics signal from the 1-nat clip.
# H3 lowers the decoder size weight so price features can compete with depth features.
PROBES = [
    ("H1_rep_loss_weight_0p5", ['--Models.WorldModel.RepresentationLossWeight', '0.5']),
    ("H2_free_bits_0p0",       ['--Models.WorldModel.FreeBits', '0.0']),
    ("H3_size_weight_1p0",     ['--Models.WorldModel.Decoder.SizeWeight', '1.0']),
]

def run_train(steps, extra_args, label):
    print(f"\n{'='*72}\n{label}\n{'='*72}")
    cmd = [
        sys.executable, '-u', '-B', 'src/train_lob.py',
        '--hours-train', str(HOURS_TRAIN),
        '--hours-val', str(HOURS_VAL),
        '--JointTrainAgent.SampleMaxSteps', str(steps),
        '--Models.WorldModel.Mamba3.is_mimo', 'False',  # MIMO requires H100.
        *extra_args,
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc:
        raise subprocess.CalledProcessError(rc, cmd)

if RUN_PROBES:
    for name, extra_args in PROBES:
        run_train(PROBE_STEPS, extra_args, f"Probe {name} ({PROBE_STEPS} steps)")
else:
    run_train(MAX_STEPS, [], f"Full pretrain ({MAX_STEPS} steps)")

In [ ]:
from pathlib import Path
import os

src = Path(PROJECT_DIR) / "saved_models" / "lob"
if not src.exists():
    print("No checkpoint directory found yet:", src)
else:
    hf_token = None
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    if not hf_token:
        hf_token = os.environ.get("HF_TOKEN")
    if hf_token and HF_REPO:
        from huggingface_hub import HfApi
        HfApi().upload_folder(
            folder_path=str(src),
            path_in_repo="checkpoints/lob",
            repo_id=HF_REPO,
            repo_type="dataset",
            token=hf_token,
        )
        print("Backed up checkpoints to HF Hub:", HF_REPO)
    else:
        print("HF_TOKEN not set - checkpoints only saved locally at:", src)
        print("Download them via the Colab Files panel before ending the session.")
